Recommendation System

Data Description:

Unique ID of each anime.\
Anime title.\
Anime broadcast type, such as TV, OVA, etc.\
anime genre.\
The number of episodes of each anime.\
The average rating for each anime compared to the number of users who gave ratings.


Number of community members for each anime.\
Objective:\
The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset. \
Dataset:\
Use the Anime Dataset which contains information about various anime, including their titles, genres,No.of episodes and user ratings etc.

Tasks:

Data Preprocessing:

Load the dataset into a suitable data structure (e.g., pandas DataFrame).\
Handle missing values, if any.\
Explore the dataset to understand its structure and attributes.

Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).\
Convert categorical features into numerical representations if necessary.\
Normalize numerical features if required.

Recommendation System:

Design a function to recommend anime based on cosine similarity.\
Given a target anime, recommend a list of similar anime based on cosine similarity scores.\
Experiment with different threshold values for similarity scores to adjust the recommendation list size.\
Analyze the performance of the recommendation system and identify areas of improvement.

Interview Questions:
1. Can you explain the difference between user-based and item-based collaborative filtering?
2. What is collaborative filtering, and how does it work?
    

## Importing Required Libraries

In [4]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

## **Data Preprocessing:**

In [5]:
# ------------------------------------------------------------
# Load the Dataset
# ------------------------------------------------------------

data = pd.read_csv("anime.csv")

print("\nFirst 5 rows of the dataset:")
print(data.head())

print("\nDataset Information:")
print(data.info())

print("\nSummary Statistics:")
print(data.describe(include="all"))

# ------------------------------------------------------------
# Handle Missing Values
# ------------------------------------------------------------

print("\nMissing Values (Before Handling):")
print(data.isnull().sum())

# Convert episodes to numeric (handles 'Unknown')
data["episodes"] = pd.to_numeric(data["episodes"], errors="coerce")

# Numerical columns
numerical_cols = ["episodes", "rating", "members"]
for col in numerical_cols:
    data[col].fillna(data[col].median())

# Categorical columns
categorical_cols = ["name", "genre", "type"]
for col in categorical_cols:
    data[col].fillna(data[col].mode()[0])

print("\nMissing Values (After Handling):")
print(data.isnull().sum())



First 5 rows of the dataset:
   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (tota

## **Feature Extraction:**

In [16]:
cat_cols = data.select_dtypes(include=['object']).columns
num_cols = data.select_dtypes(include=['int64', 'float64']).columns

cat_cols, num_cols

(Index(['name', 'genre', 'type'], dtype='object'),
 Index(['anime_id', 'episodes', 'rating', 'members'], dtype='object'))

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

data[cat_cols] = data[cat_cols].fillna("")

tfidf = TfidfVectorizer(stop_words="english")
cat_features = tfidf.fit_transform(data[cat_cols[0]])
cat_features

genre_tfidf = tfidf.fit_transform(data["genre"])
genre_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 40418 stored elements and shape (12294, 46)>

In [22]:
from sklearn.impute import SimpleImputer

num_cols = data.select_dtypes(include=["int64", "float64"]).columns

imputer = SimpleImputer(strategy="median")
data[num_cols] = imputer.fit_transform(data[num_cols])
data[num_cols]

,anime_id,episodes,rating,members
0,0.934948,0.000000,0.924370,0.197872
1,0.148091,0.034673,0.911164,0.782770
2,0.839252,0.027518,0.909964,0.112689
3,0.267972,0.012658,0.900360,0.664325
4,0.288710,0.027518,0.899160,0.149186
...,...,...,...,...
12289,0.269797,0.000000,0.297719,0.000203
12290,0.160517,0.000000,0.313325,0.000176
12291,0.162776,0.001651,0.385354,0.000211
12292,0.177605,0.000000,0.397359,0.000168


In [23]:
from sklearn.preprocessing import MinMaxScaler


scaler = MinMaxScaler()
data[num_cols] = scaler.fit_transform(data[num_cols])
data[num_cols]

,anime_id,episodes,rating,members
0,0.934948,0.000000,0.924370,0.197872
1,0.148091,0.034673,0.911164,0.782770
2,0.839252,0.027518,0.909964,0.112689
3,0.267972,0.012658,0.900360,0.664325
4,0.288710,0.027518,0.899160,0.149186
...,...,...,...,...
12289,0.269797,0.000000,0.297719,0.000203
12290,0.160517,0.000000,0.313325,0.000176
12291,0.162776,0.001651,0.385354,0.000211
12292,0.177605,0.000000,0.397359,0.000168


In [24]:
from scipy.sparse import hstack

feature_matrix = hstack([
    genre_tfidf,
    data[num_cols].values
])
feature_matrix

<COOrdinate sparse matrix of dtype 'float64'
	with 83914 stored elements and shape (12294, 50)>

In [25]:
import numpy as np

assert not np.isnan(feature_matrix.toarray()).any(), "NaNs still present!"


## **Recommendation System:**

In [26]:

# ------------------------------------------------------------
# Compute Cosine Similarity Matrix
# ------------------------------------------------------------
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(feature_matrix)

# ------------------------------------------------------------
# Recommendation Function
# ------------------------------------------------------------

def recommend_anime(anime_title, top_n=5, similarity_threshold=0.3):
    """
    Recommend anime based on cosine similarity.

    Parameters:
    anime_title (str): Target anime name
    top_n (int): Number of recommendations
    similarity_threshold (float): Minimum similarity score

    Returns:
    DataFrame of recommended anime
    """

    if anime_title not in data["name"].values:
        return "Anime title not found in the dataset."

    idx = data[data["name"] == anime_title].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Filter by threshold and remove the anime itself
    sim_scores = [
        (i, score) for i, score in sim_scores
        if score >= similarity_threshold and i != idx]

    sim_scores = sim_scores[:top_n]
    anime_indices = [i[0] for i in sim_scores]

    return data.loc[anime_indices, ["name", "genre", "type", "rating", "episodes"]]

# ------------------------------------------------------------
# Experimenting with Different Threshold Values
# ------------------------------------------------------------

print("\nRecommendations with threshold = 0.4:")
print(recommend_anime(data["name"].iloc[0], similarity_threshold=0.4))

print("\nRecommendations with threshold = 0.6:")
print(recommend_anime(data["name"].iloc[0], similarity_threshold=0.6))


Recommendations with threshold = 0.4:
                                        name  \
1111   Aura: Maryuuin Kouga Saigo no Tatakai   
208            Kokoro ga Sakebitagatterunda.   
1494                                Harmonie   
11061                           Renai Boukun   
4219             Rokujouma no Shinryakusha!?   

                                              genre     type    rating  \
1111   Comedy, Drama, Romance, School, Supernatural    Movie  0.720288   
208                          Drama, Romance, School    Movie  0.798319   
1494                    Drama, School, Supernatural    Movie  0.702281   
11061         Comedy, Romance, School, Supernatural       TV  0.588235   
4219          Comedy, Romance, School, Supernatural  Special  0.613445   

       episodes  
1111    0.00000  
208     0.00000  
1494    0.00000  
11061   0.00055  
4219    0.00000  

Recommendations with threshold = 0.6:
                                        name  \
1111   Aura: Maryuuin Kouga Saig

## **Interview Questions:**

#### 1. Difference between User-based and Item-based Collaborative Filtering
User-Based Collaborative Filtering:
- Finds users with similar preferences.
- Recommends items liked by similar users.
- Scalability issues with large user bases.

Item-Based Collaborative Filtering:
- Finds similarity between items.
- Recommends items similar to those the user liked.
- More stable and scalable.

### 2. Collaborative Filtering:
Collaborative Filtering:
- Recommendation approach based on user behavior.
- Uses ratings, clicks, or interactions.
- Assumes similar users have similar preferences.

How It Works:
- Build a user-item interaction matrix.
- Compute similarity between users or items.
- Generate recommendations based on similarity.